In [4]:
import pandas as pd
import numpy as np
import ConsultasBD
import MuestreoEInterpolacion
import FiltroButterworth
import MediaMovil
import Metricas
#importlib.reload(ConsultasBD, MuestreoEInterpolacion)

In [5]:
#Datos del usuario que se quiere reprocesar
usuarioId = 6
numeroPruebas = 1
fmuestreo = 100
fcorte = 10
ventanaMediaMovil = 100 # la ventana es de 100 muestras que equivale a un segundo

#Nombres de las tablas de bases de datos
DB_LecturasBalanceBoard = 'LecturasBalanceBoard'
DB_LecturasInterpoladas = 'LecturasInterpoladas'
DB_LecturasConFiltro = 'LecturasConFiltro'
DB_LecturasConMediaMovil = 'LecturasConMediaMovil'
DB_Evaluaciones = 'Evaluaciones'

In [6]:
def EliminarPrimerYUltimoSegundo(df, ventana):
    # Eliminar la primeras 'ventana' filas (por NaN). Estas tienen Nan por aplicar la media móvil
    # si la media móvil es de 100 muestras, que equivalen a 1 segundos, entonces las primeras 100 muestras
    # tienen valor Nan
    df = df.dropna().reset_index(drop=True)

    #df = df.iloc[:ventana].reset_index(drop=True) #Puede que con esto se soluciones
    df = df.iloc[:-ventana].reset_index(drop=True)
    print(f"✅ Media móvil aplicada. Se eliminaron el primer y último segundo ({ventana} muestras cada uno).")
    print(f"Total de muestras resultantes: {len(df)}")
    return df

if __name__ == "__main__":
    dfDatosReales = ConsultasBD.ObtenerLecturasBalanceDeBD(usuarioId, numeroPruebas)
    dfDatosMuestreados = MuestreoEInterpolacion.Interpolar(dfDatosReales, fs = fmuestreo)
    dfDatosConFiltro = FiltroButterworth.Aplicar_filtro_butterworth(dfDatosMuestreados, fc = fcorte, fs = fmuestreo)
    dfDatosMediaMovil = MediaMovil.Aplicar_media_movil(dfDatosConFiltro, ventana = ventanaMediaMovil)
    dfDatosMediaMovil = EliminarPrimerYUltimoSegundo(dfDatosMediaMovil, ventana = ventanaMediaMovil)
    metricas = Metricas.Calcular_metricas(dfDatosMediaMovil, usuarioId=usuarioId, numeroPruebas=numeroPruebas)

    # Guardar cada etapa en su tabla correspondiente
    ConsultasBD.Insertar_dataframe(dfDatosMuestreados, DB_LecturasInterpoladas, usuarioId, numeroPruebas)
    ConsultasBD.Insertar_dataframe(dfDatosConFiltro, DB_LecturasConFiltro, usuarioId, numeroPruebas)
    ConsultasBD.Insertar_dataframe(dfDatosMediaMovil, DB_LecturasConMediaMovil, usuarioId, numeroPruebas)
    ConsultasBD.Actualizar_tiempos_balanceboard(dfDatosReales, DB_LecturasBalanceBoard)

    # Guardar métricas finales
    ConsultasBD.Insertar_metricas(metricas)

c:\Users\alexs\Desktop\Preprocesamiento\ConsultasBD.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query_ConsultaLecturasBalance, conn, parse_dates=['TimeStamp'])


Filas leídas: 2286
Primeras 10 filas:
      Id  UsuarioId  NumeroPruebas    TopLeft  TopRight  BottomLeft  \
0  11105          6              1  15.901423  2.427196   -1.751574   
1  11106          6              1  15.901423  2.427196   -1.751574   
2  11107          6              1  29.270536  5.355241    7.434459   
3  11108          6              1  29.270536  5.355241    7.434459   
4  11109          6              1  28.041620  4.392068    6.655982   
5  11110          6              1  24.131435  3.621530    5.760733   
6  11111          6              1  24.131435  3.621530    5.760733   
7  11112          6              1  21.189486  2.196034    7.901546   
8  11113          6              1  23.535597  2.311615   10.431597   
9  11114          6              1  23.535597  2.311615   10.431597   

   BottomRight     COP_X     COP_Y      Total               TimeStamp  
0    -0.192090 -0.156343  0.148470   4.096239 2025-11-11 18:04:56.093  
1    -0.192090 -0.156343  0.148470  

KeyboardInterrupt: 